# Baseline gỡ bỏ học máy — NegGrad+ va CF-k tren MIMIC 3/6/10%

Doi `JOB` o Cell 2 roi Run All. Moi lan chay DUNG mot job.

| JOB | Phuong phap | Muc quen |
|---|---|---|
| `ng3`  | NegGrad+ | 3%  |
| `ng6`  | NegGrad+ | 6%  |
| `ng10` | NegGrad+ | 10% |
| `cf3`  | CF-k     | 3%  |
| `cf6`  | CF-k     | 6%  |
| `cf10` | CF-k     | 10% |

## Hai baseline nay lam gi

**NegGrad+** — tinh chinh tren `D_r` dong thoi DAO gradient tren `D_f`
(`loss = CE(retain) - lambda*CE(forget)`, lambda = |Df|/|Dr| theo chuan).
Dai dien nhom *modality-agnostic*: khong dung thong tin da phuong thuc.

**CF-k** — dong bang 2 tang dau cua MOI encoder, tinh chinh phan con lai tren `D_r`.
Dai dien nhom *han che tham so duoc cap nhat* — cung ho voi phuong phap de xuat,
chi khac cach han che.

## LUU Y BAT BUOC khi bao cao

Hai baseline nay `optimizer.step()` MOI batch tren TOAN BO retain, tuc khoang
**9.400-10.200 lan cap nhat** moi run, so voi **30** cua Forget-MI va phuong phap de
xuat. Day la cach chung duoc THIET KE de chay (va la cach bai bao goc chay), nhung:

- **KHONG** duoc dua chung vao bang chi phi tai nguyen — so sanh thoi gian vo nghia.
- Neu chung VAN khong quen duoc du co gap ~330 lan so buoc cap nhat, ket luan cang manh.

Cot `optimizer_updates` trong CSV ghi lai con so nay.


In [ ]:
# Cell 1: setup + CHOT CHAN code da push
import os, subprocess
WORK='/kaggle/working'; REPO=f'{WORK}/Forget-MI-LoKU'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/nhnhu146/Forget-MI-LoKU.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
os.chdir(REPO)
assert os.path.exists('training/adv_common.py'),'push code truoc + re-import notebook'
_adv=open('training/adv_common.py').read()
assert 'ce_selector' in _adv and 'checkpoint_selection_' in _adv, \
    '❌ adv_common CHUA co hook CE-selector -> chay `git push` code MOI roi moi Save Version!'
assert 'OnlineCESelector' in open('training/ce_selector_pilot.py').read(), '❌ git push code moi truoc!'
assert os.path.exists('training/forgetmi_p3_cand.py'), '❌ chua push forgetmi_p3_cand.py!'
print('✅ Code CE-selector da co (hook + OnlineCESelector).')
subprocess.run(['pip','install','-q','pydicom','scikit-image','scikit-learn','pyyaml','wandb','seaborn==0.13.2'],check=True)
subprocess.run(['pip','install','-q','transformers==4.38.0','peft==0.10.0','accelerate==0.27.0'],check=True)
import torch; assert torch.cuda.is_available(),'Bat GPU'
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('GPU   :',torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: CHON JOB + path discovery
import glob, os
JOB = 'ng3'      # ng3 | ng6 | ng10 | cf3 | cf6 | cf10
SEED, EPOCHS = 42, 30

JOBS = {'ng3': ('neggrad', 3), 'ng6': ('neggrad', 6), 'ng10': ('neggrad', 10),
        'cf3': ('cfk', 3),     'cf6': ('cfk', 6),     'cf10': ('cfk', 10)}
assert JOB in JOBS, f'JOB phai thuoc {sorted(JOBS)}'
METHOD, PCT = JOBS[JOB]

def fd(*slugs):
    for s in slugs:
        if os.path.isdir(f'/kaggle/input/{s}'): return f'/kaggle/input/{s}'
        h = glob.glob(f'/kaggle/input/datasets/*/{s}')
        if h: return sorted(h)[0]
    return None
def bins(root): return sorted(glob.glob(os.path.join(root,'**','pytorch_model.bin'),recursive=True),key=len)

DATA = fd('forget-mi-data'); MOD = fd('forget-mi-models-full','forget-mi-models')
assert DATA and MOD, 'Chua Add Input: forget-mi-data + forget-mi-models-full'
BASE = os.path.dirname([b for b in bins(MOD) if 'training_original_model' in b][0])
gh = [b for b in bins(MOD) if f'model_retrained_{PCT}per' in b]
assert gh, f'Khong thay model_retrained_{PCT}per'
GOLD = os.path.dirname(gh[0])

RID = f'{JOB}_s{SEED}'
OUT = f'/kaggle/working/base_{RID}'
RESULTS = f'/kaggle/working/results_baselines.csv'   # DUNG CHUNG cho ca 6 run
OVR = {'forget_set_path': f'./data_splits/forget_set_{PCT}per.csv',
       'base_model_path': BASE, 'bert_pretrained_dir': BASE,
       'retrained_model_path': GOLD,
       'text_data_dir': os.path.join(DATA,'data','metadata'),
       'img_data_dir': os.path.join(DATA,'data','img_data'),
       'data_split_path': './data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv',
       'results_csv_path': RESULTS, 'output_dir': OUT, 'id': RID}

# Chi kiem tra duong dan DAU VAO. output_dir / results_csv_path la dau RA nen
# chua ton tai la binh thuong — dua chung vao danh sach kiem tra se chan moi run.
INPUTS = ('forget_set_path','base_model_path','bert_pretrained_dir',
          'retrained_model_path','text_data_dir','img_data_dir','data_split_path')
print('=== kiem tra duong dan dau vao ===')
for k, v in OVR.items():
    mark = (' OK' if os.path.exists(str(v)) else '  *** THIEU ***') if k in INPUTS else '  (dau ra)'
    print(f'   {k:22} {v}{mark}')
miss = [k for k in INPUTS if not os.path.exists(str(OVR[k]))]
assert not miss, f'Thieu duong dan dau vao: {miss}'
print()
print(f'JOB {JOB} -> method={METHOD}  forget={PCT}%  epochs={EPOCHS}  run id={RID}')


In [ ]:
# Cell 3: CHAY
import os, subprocess, time
env = {**os.environ, 'PYTHONPATH': '.', 'WANDB_MODE': 'disabled',
       'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}
cmd = ['python', 'scripts/unlearn_baselines.py',
       '--config', 'config_baseline_kaggle.yaml',
       '--method', METHOD, '--seed', str(SEED), '--epochs', str(EPOCHS),
       '--override', ','.join(f'{k}={v}' for k, v in OVR.items())]
print('='*72 + f'\n{RID}   method={METHOD}  forget={PCT}%\n' + '='*72)
t0 = time.time()
try:
    subprocess.run(cmd, env=env, check=True)
    print(f'OK {RID}  wall {(time.time()-t0)/3600:.2f}h')
except subprocess.CalledProcessError as e:
    print('FAIL', RID, 'rc=', e.returncode)


In [ ]:
# Cell 4: xem ket qua
import os, pandas as pd
pd.set_option('display.width', 220)
if os.path.exists(RESULTS):
    d = pd.read_csv(RESULTS)
    cols = [c for c in ['id','method','checkpoint','Df_AUC','Df_F1','Dt_AUC','Dt_F1',
                        'MIA','MIA_paper','forget_ce','test_ce','trainable_ratio',
                        'optimizer_updates'] if c in d.columns]
    print(d[cols].to_string(index=False))
    print('''
Doc ket qua — moc de doi chieu (da co san trong Chuong 4):
  MIMIC 3%   theta_og  Df-AUC 0.731  MIA 0.657   |  theta_re  Df-AUC 0.498  MIA 0.423
  MIMIC 6%   theta_og  Df-AUC 0.730  MIA 0.736   |  theta_re  Df-AUC 0.596  MIA 0.613
  MIMIC 10%  theta_og  Df-AUC 0.757  MIA 0.717   |  theta_re  Df-AUC 0.547  MIA 0.364

Baseline SAT theta_og  -> khong quen duoc (dung nhu bai bao goc bao cao).
Baseline VUOT theta_re -> quen qua da, mo hinh hong.''')
else:
    print('chua co', RESULTS)
print('\nTAI VE: results_baselines.csv (gom du ca 6 run neu chay chung 1 session)')
